# Notebook 01b — Core vs Module Questions and Variable Dictionary

### Why this notebook exists
BRFSS has two kinds of questions:
- **Core** — asked in **every** state, to every eligible respondent.
- **Optional module** — each state chooses which modules to run, so most respondents were **never asked** these questions.

After cleaning, 66% of feature cells are blank — mostly because of optional modules, not because people refused to answer.
Imputing "never asked" as if it were a real answer is wrong (reviewer comment #7), and it let the old model learn
*who was asked* instead of *what they answered*.

### What this notebook does
1. Downloads the official BRFSS 2023 codebook from the CDC and reads every variable's label and Core/Module tag
2. Checks the tag against the data — in how many states was each question actually asked?
3. Compares missingness of core vs module questions, overall and by stroke status
4. Saves `outputs/variable_dictionary.csv` — one row per feature with its codebook label, type and missingness
   (this is the documented feature table reviewer comment #13 asks for)

**Input** — `data/processed/brfss_cleaned_v2.csv` (from Notebook 01 v2)
**Output** — `outputs/variable_dictionary.csv`, `outputs/missingness_core_vs_module.csv`

In [ ]:
import pandas as pd
import numpy as np
import os, re, html, zipfile, urllib.request
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR      = '/content/drive/MyDrive/stroke_prediction'
CLEANED_PATH  = os.path.join(BASE_DIR, 'data', 'processed', 'brfss_cleaned_v2.csv')
CODEBOOK_DIR  = os.path.join(BASE_DIR, 'data', 'raw', 'codebook23')
OUTPUTS_DIR   = os.path.join(BASE_DIR, 'outputs')
DICT_PATH     = os.path.join(OUTPUTS_DIR, 'variable_dictionary.csv')
SUMMARY_PATH  = os.path.join(OUTPUTS_DIR, 'missingness_core_vs_module.csv')
os.makedirs(CODEBOOK_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print("Loading cleaned dataset...")
df = pd.read_csv(CLEANED_PATH)
META_COLS    = [c for c in df.columns if c.startswith('meta_')]
FEATURE_COLS = [c for c in df.columns if c != 'stroke' and c not in META_COLS]
print(f"✓ Loaded: {len(df):,} respondents | {len(FEATURE_COLS)} features | {len(META_COLS)} meta columns")

### Step 1 — Get the official BRFSS 2023 codebook

Downloaded once from the CDC and stored in `data/raw/codebook23/`, so the analysis is reproducible.
This is the same codebook cited in the paper (USCODE23_LLCP_021924.HTML).

In [ ]:
CODEBOOK_URL  = 'https://www.cdc.gov/brfss/annual_data/2023/zip/codebook23_llcp-v2-508.zip'
CODEBOOK_HTML = os.path.join(CODEBOOK_DIR, 'USCODE23_LLCP_021924.HTML')

if not os.path.exists(CODEBOOK_HTML):
    zip_path = os.path.join(CODEBOOK_DIR, 'codebook23_llcp-v2-508.zip')
    print("Downloading codebook from the CDC...")
    urllib.request.urlretrieve(CODEBOOK_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(CODEBOOK_DIR)

print("✓ Codebook ready:", CODEBOOK_HTML)

In [ ]:
# ── Parse the codebook: label, section and Core/Module tag for every variable ──
def parse_codebook(path):
    text = open(path, encoding='latin1').read()
    text = html.unescape(text).replace('\xa0', ' ')
    blocks = re.split(r'(?=Label:\s)', text)   # each variable block starts at "Label:"
    out = {}
    for b in blocks:
        name = re.search(r'SAS Variable Name:\s*([A-Z0-9_]+)', b)
        if not name:
            continue
        label   = re.search(r'Label:\s*([^<|]+)', b)
        section = re.search(r'Section Name:\s*([^<]+)', b)
        kind    = re.search(r'(Core) Section Number|(Module) Number', b)
        out[name.group(1)] = {
            'label'  : label.group(1).strip() if label else '',
            'section': section.group(1).strip() if section else '',
            'kind'   : (kind.group(1) or kind.group(2)) if kind else '',
        }
    return out

codebook = parse_codebook(CODEBOOK_HTML)
print(f"✓ Parsed {len(codebook)} variables from the codebook")
print(f"   Core   : {sum(v['kind'] == 'Core' for v in codebook.values())}")
print(f"   Module : {sum(v['kind'] == 'Module' for v in codebook.values())}")

### Step 2 — Tag each feature as Core or Module

Some columns were renamed in Notebook 01 (e.g. `_AGE80` → `age`), so they are mapped back to their codebook name.
`age` and `bmi` are CDC-calculated from core questions, and `SEXVAR` is part of the core interview, so all three count as Core.

In [ ]:
RENAMED = {
    'bmi'                   : '_BMI5',
    'age'                   : '_AGE80',
    'gender'                : 'SEXVAR',
    'hypertension'          : 'BPHIGH6',
    'diabetes_status'       : 'DIABETE4',
    'alcohol_days_per_month': 'ALCDAY4',
}
FORCE_CORE = {'_BMI5', '_AGE80', 'SEXVAR'}   # calculated from / part of the core interview

rows = []
for col in FEATURE_COLS:
    cb_name = RENAMED.get(col, col)
    info = codebook.get(cb_name, {'label': '', 'section': '', 'kind': ''})
    kind = 'Core' if cb_name in FORCE_CORE else info['kind']
    rows.append({
        'feature'      : col,
        'codebook_name': cb_name,
        'label'        : info['label'],
        'section'      : info['section'],
        'type'         : kind if kind else 'Unknown',
    })
var_dict = pd.DataFrame(rows)

print(var_dict['type'].value_counts().to_string())
unknown = var_dict.loc[var_dict['type'] == 'Unknown', 'feature'].tolist()
print(f"\nNot found in codebook: {len(unknown)} {unknown}")

### Step 3 — Check the tags against the data: in how many states was each question asked?

A core question should have answers from (almost) every state. A module question should only have answers from the
states that chose that module. This confirms the codebook tag using the data itself.

In [ ]:
n_states = df['meta_state'].nunique()

# For each feature: number of states with at least one non-missing answer
answered_by_state = df[FEATURE_COLS].notna().groupby(df['meta_state']).any()
var_dict['states_asked'] = var_dict['feature'].map(answered_by_state.sum())
var_dict['pct_states']   = (var_dict['states_asked'] / n_states * 100).round(1)

print(f"States / territories in the data: {n_states}\n")
print("States asked — summary by type:")
print(var_dict.groupby('type')['states_asked'].agg(['count', 'min', 'median', 'max']).to_string())

# Flag disagreements between the codebook tag and the data
odd_core   = var_dict[(var_dict['type'] == 'Core')   & (var_dict['pct_states'] < 90)]
odd_module = var_dict[(var_dict['type'] == 'Module') & (var_dict['pct_states'] >= 90)]
print(f"\nCore questions asked in <90% of states   : {len(odd_core)}")
if len(odd_core):
    print(odd_core[['feature', 'section', 'states_asked']].to_string(index=False))
print(f"Module questions asked in >=90% of states: {len(odd_module)}")
if len(odd_module):
    print(odd_module[['feature', 'section', 'states_asked']].to_string(index=False))

### Step 4 — Missingness: core vs module, and stroke vs no stroke

`pct_missing_gap` = missing % among stroke respondents minus missing % among non-stroke respondents.
A large gap means **whether the question is blank** already carries information about stroke — a warning sign that
the model could learn the survey's routing instead of health information.

In [ ]:
is_stroke = df['stroke'] == 1
miss = df[FEATURE_COLS].isna()

var_dict['pct_missing']        = (miss.mean() * 100).round(1).values
var_dict['pct_missing_stroke'] = (miss[is_stroke].mean() * 100).round(1).values
var_dict['pct_missing_no_str'] = (miss[~is_stroke].mean() * 100).round(1).values
var_dict['pct_missing_gap']    = (var_dict['pct_missing_stroke'] - var_dict['pct_missing_no_str']).round(1)

summary = var_dict.groupby('type').agg(
    n_features        = ('feature', 'count'),
    mean_pct_missing  = ('pct_missing', 'mean'),
    median_pct_missing= ('pct_missing', 'median'),
    mean_abs_gap      = ('pct_missing_gap', lambda s: s.abs().mean()),
).round(1)
print("=== Missingness by question type ===")
print(summary.to_string())
summary.to_csv(SUMMARY_PATH)

# Share of blank answers per respondent, core questions only
core_cols = var_dict.loc[var_dict['type'] == 'Core', 'feature'].tolist()
core_missing_pct = df[core_cols].isna().mean(axis=1)
print(f"\n=== Per-respondent missingness, CORE questions only ({len(core_cols)} features) ===")
print(core_missing_pct.describe(percentiles=[.5, .9, .99]).round(3).to_string())

print(f"\n{'Threshold':>10} {'Removed stroke=1':>17} {'Removed stroke=0':>17} {'Rows left':>10} {'Prevalence left':>16}")
for t in [None, 0.5, 0.3, 0.2]:
    keep  = pd.Series(True, index=df.index) if t is None else core_missing_pct <= t
    label = 'none' if t is None else f'>{int(t*100)}%'
    removed, kept = df.loc[~keep, 'stroke'], df.loc[keep, 'stroke']
    print(f"{label:>10} {int((removed==1).sum()):>17,} {int((removed==0).sum()):>17,} "
          f"{len(kept):>10,} {kept.mean()*100:>15.2f}%")

In [ ]:
# ── Core questions with high missingness = skip patterns (asked only if a previous answer qualifies) ──
core_view = (var_dict[var_dict['type'] == 'Core']
             .sort_values('pct_missing', ascending=False)
             [['feature', 'section', 'pct_missing', 'pct_missing_gap', 'label']])
print("=== Core questions, most missing first ===")
print("(high % missing in a core question usually means a skip pattern, e.g. asked only of smokers)\n")
with pd.option_context('display.max_colwidth', 60, 'display.width', 200):
    print(core_view.head(25).to_string(index=False))

print("\n=== Module questions with the largest stroke vs no-stroke missingness gap ===")
mod_view = (var_dict[var_dict['type'] == 'Module']
            .assign(abs_gap=lambda d: d['pct_missing_gap'].abs())
            .sort_values('abs_gap', ascending=False)
            [['feature', 'section', 'pct_missing', 'pct_missing_gap']])
print(mod_view.head(15).to_string(index=False))

### Step 5 — Save the variable dictionary

In [ ]:
var_dict = var_dict.sort_values(['type', 'section', 'feature']).reset_index(drop=True)
var_dict.to_csv(DICT_PATH, index=False)

print("=" * 60)
print("  Notebook 01b complete")
print("=" * 60)
print(f"  Features            : {len(var_dict)}")
for t, n in var_dict['type'].value_counts().items():
    print(f"    {t:<8}: {n}")
print(f"  Saved dictionary    : {DICT_PATH}")
print(f"  Saved summary       : {SUMMARY_PATH}")
print("=" * 60)

### How to read the results

- **Core questions** should show low missingness and a small stroke/no-stroke gap → safe to impute and model.
- **Core questions with high missingness** are skip patterns (e.g. "how often do you smoke?" asked only of smokers).
  These need a logical fill (e.g. never-smoker → 0) or a missing-indicator, not mean imputation.
- **Module questions** are blank for most respondents because their state did not ask them. The main model will use
  core questions only; module questions can be tested separately as a sensitivity analysis.